# Importing Libraries

In [1]:
import pandas as pd
import re
from pathlib import Path
from datetime import datetime, timezone, timedelta

# Setup Configuration

## Load Paths

In [43]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"
OUTPUT_PATH = DATA_DIR / "jobs_enriched.csv"

COMPANIES_PATH = DATA_DIR / "companies.csv"

JOBS_RAW_PATH = DATA_DIR / "jobs_raw.csv"



print(f"Base directory: {BASE_DIR} | Base directory exists: {BASE_DIR.exists()}")
print(f"Companies path: {COMPANIES_PATH} |File exists: {COMPANIES_PATH.exists()}")
print(f"Jobs Raw Path: {JOBS_RAW_PATH} | Jobs Raw directory exists: {JOBS_RAW_PATH.parent.exists()}")

Base directory: c:\Users\surab\OneDrive\Documents\Personal\Projects\jobscout-ai | Base directory exists: True
Companies path: c:\Users\surab\OneDrive\Documents\Personal\Projects\jobscout-ai\data\companies.csv |File exists: True
Jobs Raw Path: c:\Users\surab\OneDrive\Documents\Personal\Projects\jobscout-ai\data\jobs_raw.csv | Jobs Raw directory exists: True


## Load Registry


In [7]:
companies_df = pd.read_csv(COMPANIES_PATH).fillna("")
print("Companies loaded:", len(companies_df))
# companies_df.head()

Companies loaded: 5


In [8]:


jobs_df = pd.read_csv(JOBS_RAW_PATH).fillna("")

print("Jobs loaded:", len(jobs_df))


jobs_df.head()

Jobs loaded: 1265


,company,title,location,job_url,description,ats_type,external_job_id,posted_date,date_found,description_raw,remote_type,posted_datetime,freshness_status
0,OpenAI,"Technical Program Manager, Compute Infrastructure",San Francisco,https://jobs.ashbyhq.com/openai/8fb1615c-34bf-...,About the Team The compute infrastructure team...,ashby,8fb1615c-34bf-47c4-a1d1-b7b2f836bbd3,2026-03-12T16:38:15.322+00:00,2026-06-04T18:11:09.402369+00:00,,,2026-03-12 16:38:15.322000+00:00,old
1,OpenAI,Research Engineer,San Francisco,https://jobs.ashbyhq.com/openai/240d459b-696d-...,"By applying to this role, you will be consider...",ashby,240d459b-696d-43eb-8497-fab3e56ecd9b,2025-04-05T00:03:20.653+00:00,2026-06-04T18:11:09.407935+00:00,,,2025-04-05 00:03:20.653000+00:00,old
2,OpenAI,Account Director - Tokyo,"Tokyo, Japan",https://jobs.ashbyhq.com/openai/18f58952-c242-...,About the team OpenAI’s mission is to build sa...,ashby,18f58952-c242-4562-8732-073a0ae8029e,2026-01-23T00:17:18.483+00:00,2026-06-04T18:11:09.410524+00:00,,,2026-01-23 00:17:18.483000+00:00,old
3,OpenAI,"Software Engineer, RL Training Infra",San Francisco,https://jobs.ashbyhq.com/openai/13995549-e8cc-...,About the Team The Post-Training Frontiers tea...,ashby,13995549-e8cc-498f-9eaa-1869067ac35b,2026-05-23T02:00:50.464+00:00,2026-06-04T18:11:09.410524+00:00,,,2026-05-23 02:00:50.464000+00:00,old
4,OpenAI,"Research Engineer, Retrieval & Search, Applied...",San Francisco,https://jobs.ashbyhq.com/openai/7322d344-9325-...,About the Team We bring OpenAI's technology to...,ashby,7322d344-9325-4a92-8445-0a2c4e9272f8,2024-03-20T21:33:20.763+00:00,2026-06-04T18:11:09.410524+00:00,,,2024-03-20 21:33:20.763000+00:00,old


## Check for missing columns

In [10]:
required_job_columns = [
    "company",
    "title",
    "location",
    "job_url",
    "description",
    "ats_type",
    "external_job_id",
    "posted_date",
    "posted_datetime",
    "freshness_status",
    "date_found"
]

missing_columns = [
    col for col in required_job_columns
    if col not in jobs_df.columns
]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All required job columns are present.")

All required job columns are present.


## Inspect freshness status

In [14]:
jobs_df["freshness_status"].value_counts()

freshness_status
old      1236
fresh      29
Name: count, dtype: int64

In [15]:
jobs_df.groupby("ats_type").agg(
    total_jobs=("title", "count"),
    fresh_jobs=("freshness_status", lambda x: (x == "fresh").sum()),
    unknown_jobs=("freshness_status", lambda x: (x == "unknown").sum()),
    old_jobs=("freshness_status", lambda x: (x == "old").sum())
)

,total_jobs,fresh_jobs,unknown_jobs,old_jobs
ats_type,,,,
ashby,713,11,0,702
greenhouse,366,5,0,361
lever,166,0,0,166
workday,20,13,0,7


## End


# Helper Functions

## Normalize keywords

In [9]:
def normalize_text(*values):
    """
    Combines multiple values into one lowercase searchable string.
    """
    return " ".join([str(v) for v in values if v is not None]).lower()

## Split Keywords

In [11]:
def split_keywords(keyword_string):
    """
    Converts semicolon-separated keywords into a clean lowercase list.
    Example:
    'machine learning;data scientist;ai engineer'
    ->
    ['machine learning', 'data scientist', 'ai engineer']
    """
    if not keyword_string:
        return []

    return [
        keyword.strip().lower()
        for keyword in str(keyword_string).split(";")
        if keyword.strip()
    ]


## Keyword match

In [12]:
def keyword_in_text(keyword, text):
    """
    Safer keyword match.
    - For multi-word phrases: normal substring match.
    - For single words: word-boundary regex, so 'api' does not match random text badly.
    """
    keyword = str(keyword).lower().strip()
    text = str(text).lower()

    if not keyword:
        return False

    if " " in keyword:
        return keyword in text

    pattern = r"\b" + re.escape(keyword) + r"\b"
    return re.search(pattern, text) is not None

In [13]:
def find_keyword_hits(text, keywords):
    """
    Returns the list of keywords found in text.
    """
    return [
        keyword
        for keyword in keywords
        if keyword_in_text(keyword, text)
    ]

## company-specific keyword matching

In [17]:
# Build company keyword map
company_keywords = {
    row["company"]: split_keywords(row["keywords"])
    for _, row in companies_df.iterrows()
}

company_keywords

{'OpenAI': ['machine learning',
  'data scientist',
  'ai engineer',
  'software engineer'],
 'Anthropic': ['machine learning',
  'data scientist',
  'ai engineer',
  'software engineer'],
 'Netflix': ['machine learning', 'data scientist', 'software engineer'],
 'Workday': ['machine learning', 'data scientist', 'software engineer'],
 'Spotify': ['machine learning', 'data scientist', 'software engineer', 'ai']}

In [18]:
def match_company_keywords(row):
    company = row.get("company", "")
    keywords = company_keywords.get(company, [])

    searchable_text = normalize_text(
        row.get("title", ""),
        row.get("description", ""),
        row.get("location", "")
    )

    return find_keyword_hits(searchable_text, keywords)

## Profile Keywords

In [21]:
target_role_keywords = [
    "machine learning engineer",
    "ml engineer",
    "ai engineer",
    "applied ai",
    "applied scientist",
    "data scientist",
    "data science",
    "ml data engineer",
    "analytics engineer",
    "data engineer",
    "knowledge graph",
    "graph machine learning",
    "research engineer",
    "computer vision engineer",
    "nlp engineer",
    "software engineer",
    "ai data scientist"
]

In [22]:
target_skill_keywords = [
    "python",
    "sql",
    "machine learning",
    "deep learning",
    "pytorch",
    "tensorflow",
    "scikit-learn",
    "sklearn",
    "pandas",
    "numpy",
    "llm",
    "large language model",
    "generative ai",
    "computer vision",
    "nlp",
    "transformers",
    "hugging face",
    "data pipeline",
    "etl",
    "airflow",
    "docker",
    "aws",
    "azure",
    "gcp",
    "postgresql",
    "neo4j",
    "graph",
    "knowledge graph",
    "flask",
    "api",
    "rest",
    "model evaluation",
    "experimentation"
]

In [23]:
project_relevance_keywords = [
    "computer vision",
    "image",
    "classification",
    "detection",
    "llm",
    "generative ai",
    "graph",
    "knowledge graph",
    "fraud",
    "anomaly detection",
    "data pipeline",
    "machine learning",
    "research",
    "nlp"
]

In [25]:
senior_level_keywords = [
    "senior",
    "staff",
    "principal",
    "lead",
    "manager",
    "director",
    "head of",
    "vp",
    "vice president",
    "10+ years",
    "8+ years",
    "7+ years",
    "6+ years",
    "5+ years"
]

In [26]:
good_level_keywords = [
    "new grad",
    "early career",
    "entry level",
    "associate",
    "junior",
    "university graduate",
    "graduate",
    "0+ years",
    "1+ years",
    "2+ years",
]

## Role Score

In [30]:
def calculate_role_score(row):
    title_text = normalize_text(row.get("title", ""))
    full_text = normalize_text(
        row.get("title", ""),
        row.get("description", "")
    )

    title_role_hits = find_keyword_hits(title_text, target_role_keywords)
    all_role_hits = find_keyword_hits(full_text, target_role_keywords)

    score = 0

    if title_role_hits:
        score += 20
    elif all_role_hits:
        score += 12

    score += min(len(all_role_hits) * 2, 5)

    return min(score, 25), all_role_hits

## Skill score

In [27]:
def calculate_skill_score(row):
    full_text = normalize_text(
        row.get("title", ""),
        row.get("description", "")
    )

    skill_hits = find_keyword_hits(full_text, target_skill_keywords)

    score = len(skill_hits) * 3

    return min(score, 35), skill_hits

## Project relevance score

In [31]:
def calculate_project_relevance_score(row):
    full_text = normalize_text(
        row.get("title", ""),
        row.get("description", "")
    )

    project_hits = find_keyword_hits(full_text, project_relevance_keywords)

    score = len(project_hits) * 3

    return min(score, 15), project_hits

## Experience score

In [32]:
def calculate_experience_score(row):
    title_text = normalize_text(row.get("title", ""))
    full_text = normalize_text(
        row.get("title", ""),
        row.get("description", "")
    )

    senior_hits = find_keyword_hits(full_text, senior_level_keywords)
    good_level_hits = find_keyword_hits(full_text, good_level_keywords)

    score = 10

    if good_level_hits:
        score += 5

    # Strong penalty if seniority appears in title
    title_senior_hits = find_keyword_hits(title_text, senior_level_keywords)

    if title_senior_hits:
        score -= 10
    elif senior_hits:
        score -= 6

    score = max(score, 0)

    return min(score, 15), good_level_hits, senior_hits

## Freshness Score

In [33]:
def calculate_freshness_score(row):
    freshness = str(row.get("freshness_status", "")).lower()

    if freshness == "fresh":
        return 10

    if freshness == "unknown":
        return 6

    return 0

## Required check

In [42]:
list_columns = [
    "matched_keywords",
    "matched_roles",
    "matched_skills",
    "project_relevance_hits",
    "missing_keywords",
    "seniority_flags",
    "good_level_hits"
]

for col in list_columns:
    if col in jobs_df.columns:
        jobs_df[col] = jobs_df[col].apply(
            lambda x: "; ".join(x) if isinstance(x, list) else x
        )

## Final Score

In [28]:
def calculate_ats_score(row):
    role_score, matched_roles = calculate_role_score(row)
    skill_score, matched_skills = calculate_skill_score(row)
    project_score, project_hits = calculate_project_relevance_score(row)
    experience_score, good_level_hits, seniority_flags = calculate_experience_score(row)
    freshness_score = calculate_freshness_score(row)

    ats_match_score = (
        role_score
        + skill_score
        + project_score
        + experience_score
        + freshness_score
    )

    missing_keywords = [
        keyword
        for keyword in target_skill_keywords
        if keyword not in matched_skills
    ]

    if ats_match_score >= 80:
        score_label = "Strong Match"
    elif ats_match_score >= 65:
        score_label = "Good Match"
    elif ats_match_score >= 50:
        score_label = "Maybe"
    else:
        score_label = "Low Match"

    score_reasons = []

    if matched_roles:
        score_reasons.append(
            "Role match: " + ", ".join(matched_roles[:5])
        )

    if matched_skills:
        score_reasons.append(
            "Skills matched: " + ", ".join(matched_skills[:8])
        )

    if project_hits:
        score_reasons.append(
            "Project relevance: " + ", ".join(project_hits[:5])
        )

    if seniority_flags:
        score_reasons.append(
            "Seniority concern: " + ", ".join(seniority_flags[:5])
        )

    if not score_reasons:
        score_reasons.append("Limited match based on current scoring rules.")

    return {
        "role_score": role_score,
        "skill_score": skill_score,
        "project_score": project_score,
        "experience_score": experience_score,
        "freshness_score": freshness_score,
        "ats_match_score": ats_match_score,
        "score_label": score_label,
        "matched_roles": matched_roles,
        "matched_skills": matched_skills,
        "project_relevance_hits": project_hits,
        "missing_keywords": missing_keywords[:15],
        "seniority_flags": seniority_flags,
        "good_level_hits": good_level_hits,
        "score_reason": " | ".join(score_reasons)
    }

## End

# Run 

## Keyword Match Count

In [19]:
jobs_df["matched_keywords"] = jobs_df.apply(
    match_company_keywords,
    axis=1
)

jobs_df["keyword_match_count"] = jobs_df["matched_keywords"].apply(len)

jobs_df[
    [
        "company",
        "title",
        "matched_keywords",
        "keyword_match_count"
    ]
].head(20)

,company,title,matched_keywords,keyword_match_count
0,OpenAI,"Technical Program Manager, Compute Infrastructure",[],0
1,OpenAI,Research Engineer,[machine learning],1
2,OpenAI,Account Director - Tokyo,[],0
3,OpenAI,"Software Engineer, RL Training Infra",[software engineer],1
4,OpenAI,"Research Engineer, Retrieval & Search, Applied...",[machine learning],1
5,OpenAI,"Researcher, Robustness & Safety Training",[machine learning],1
6,OpenAI,"Software Engineer, Data Infrastructure","[machine learning, software engineer]",2
7,OpenAI,Training: ML Framework Engineer,"[machine learning, software engineer]",2
8,OpenAI,"Software Engineer, Developer Productivity",[software engineer],1
9,OpenAI,"Software Engineer, Data Acquisition",[software engineer],1


## Score all jobs

In [34]:
score_results = jobs_df.apply(calculate_ats_score, axis=1)
score_df = pd.DataFrame(score_results.tolist())

jobs_df = pd.concat(
    [
        jobs_df.reset_index(drop=True),
        score_df.reset_index(drop=True)
    ],
    axis=1
)

jobs_df[
    [
        "company",
        "title",
        "location",
        "freshness_status",
        "ats_match_score",
        "score_label",
        "matched_skills",
        "seniority_flags",
        "job_url"
    ]
].sort_values("ats_match_score", ascending=False).head(25)

,company,title,location,freshness_status,ats_match_score,score_label,matched_skills,seniority_flags,job_url
1180,Spotify,"Machine Learning Engineer, Personalization, Mi...","New York, NY US remote",old,74,Good Match,"[python, sql, machine learning, pytorch, tenso...",[],https://jobs.lever.co/spotify/de3f6a47-4d75-45...
1179,Spotify,"Machine Learning Engineer I, Personalization ,...","New York, NY US remote",old,74,Good Match,"[python, sql, machine learning, pytorch, tenso...",[],https://jobs.lever.co/spotify/fd79c3f5-1b2c-47...
1231,Spotify,"Senior Machine Learning Engineer, Zeitgeist, P...","New York, NY US remote",old,70,Good Match,"[python, machine learning, llm, large language...","[senior, 5+ years]",https://jobs.lever.co/spotify/351ad979-231f-4b...
1176,Spotify,Machine Learning Engineer,"New York, NY US remote",old,70,Good Match,"[python, machine learning, pytorch, large lang...",[lead],https://jobs.lever.co/spotify/e200d025-4cff-4e...
1232,Spotify,"Senior Machine Learning Engineer, Zeitgeist, P...",London GB remote,old,70,Good Match,"[python, machine learning, llm, large language...","[senior, 5+ years]",https://jobs.lever.co/spotify/e3e57517-0677-40...
1079,Workday,Senior Software Engineer (Gen AI),"USA, CO, Boulder",fresh,70,Good Match,"[python, machine learning, llm, large language...","[senior, lead, 5+ years]",https://workday.wd5.myworkdayjobs.com/job/USA-...
310,OpenAI,Research Engineer / Machine Learning Engineer ...,San Francisco,old,67,Good Match,"[machine learning, deep learning, pytorch, ten...","[lead, head of]",https://jobs.ashbyhq.com/openai/46cd47bc-d4de-...
1255,Spotify,Staff Machine Learning Engineer,London GB remote,old,66,Good Match,"[python, machine learning, pytorch, large lang...","[staff, lead]",https://jobs.lever.co/spotify/f3616bfc-a2bb-48...
1254,Spotify,Staff Machine Learning Engineer,"New York, NY US remote",old,66,Good Match,"[python, machine learning, pytorch, large lang...","[staff, lead]",https://jobs.lever.co/spotify/736f1827-6b26-4b...
901,Anthropic,"ML Infrastructure Engineer, Safeguards","San Francisco, CA",old,63,Maybe,"[python, machine learning, pytorch, tensorflow...","[staff, 5+ years]",https://job-boards.greenhouse.io/anthropic/job...


## Relevancy tag

In [39]:
jobs_df["is_relevant"] = (
    jobs_df["freshness_status"].isin(["fresh", "unknown"])
    & (jobs_df["keyword_match_count"] > 0)
    & (jobs_df["ats_match_score"] >= 50)
)

print("Total jobs:", len(jobs_df))
print("Relevant jobs:", jobs_df["is_relevant"].sum())
print("Strong matches:", (jobs_df["ats_match_score"] >= 80).sum())

Total jobs: 1265
Relevant jobs: 1
Strong matches: 0


In [41]:
strong_matches_df = jobs_df[
    jobs_df["ats_match_score"] >= 60
].sort_values("ats_match_score", ascending=False)

strong_matches_df[
    [
        "company",
        "title",
        "location",
        "freshness_status",
        "ats_match_score",
        "score_label",
        "score_reason",
        "job_url"
    ]
].head(30)

,company,title,location,freshness_status,ats_match_score,score_label,score_reason,job_url
1179,Spotify,"Machine Learning Engineer I, Personalization ,...","New York, NY US remote",old,74,Good Match,"Role match: machine learning engineer, ml engi...",https://jobs.lever.co/spotify/fd79c3f5-1b2c-47...
1180,Spotify,"Machine Learning Engineer, Personalization, Mi...","New York, NY US remote",old,74,Good Match,"Role match: machine learning engineer, ml engi...",https://jobs.lever.co/spotify/de3f6a47-4d75-45...
1079,Workday,Senior Software Engineer (Gen AI),"USA, CO, Boulder",fresh,70,Good Match,"Role match: ml engineer, data science, data en...",https://workday.wd5.myworkdayjobs.com/job/USA-...
1176,Spotify,Machine Learning Engineer,"New York, NY US remote",old,70,Good Match,"Role match: machine learning engineer, data sc...",https://jobs.lever.co/spotify/e200d025-4cff-4e...
1232,Spotify,"Senior Machine Learning Engineer, Zeitgeist, P...",London GB remote,old,70,Good Match,"Role match: machine learning engineer, ml engi...",https://jobs.lever.co/spotify/e3e57517-0677-40...
1231,Spotify,"Senior Machine Learning Engineer, Zeitgeist, P...","New York, NY US remote",old,70,Good Match,"Role match: machine learning engineer, ml engi...",https://jobs.lever.co/spotify/351ad979-231f-4b...
310,OpenAI,Research Engineer / Machine Learning Engineer ...,San Francisco,old,67,Good Match,"Role match: machine learning engineer, researc...",https://jobs.ashbyhq.com/openai/46cd47bc-d4de-...
1254,Spotify,Staff Machine Learning Engineer,"New York, NY US remote",old,66,Good Match,"Role match: machine learning engineer, data sc...",https://jobs.lever.co/spotify/736f1827-6b26-4b...
1255,Spotify,Staff Machine Learning Engineer,London GB remote,old,66,Good Match,"Role match: machine learning engineer, data sc...",https://jobs.lever.co/spotify/f3616bfc-a2bb-48...
901,Anthropic,"ML Infrastructure Engineer, Safeguards","San Francisco, CA",old,63,Maybe,Role match: data engineer | Skills matched: py...,https://job-boards.greenhouse.io/anthropic/job...


## End


# Save CSV

In [44]:


jobs_df.to_csv(OUTPUT_PATH, index=False)

print("Saved enriched jobs to:", OUTPUT_PATH)
print("Total jobs:", len(jobs_df))
print("Relevant jobs:", jobs_df["is_relevant"].sum())
print("Strong matches:", (jobs_df["ats_match_score"] >= 80).sum())

Saved enriched jobs to: c:\Users\surab\OneDrive\Documents\Personal\Projects\jobscout-ai\data\jobs_enriched.csv
Total jobs: 1265
Relevant jobs: 1
Strong matches: 0


In [45]:
jobs_df[
    [
        "company",
        "title",
        "ats_type",
        "freshness_status",
        "keyword_match_count",
        "ats_match_score",
        "score_label",
        "is_relevant"
    ]
].sort_values("ats_match_score", ascending=False).head(20)

,company,title,ats_type,freshness_status,keyword_match_count,ats_match_score,score_label,is_relevant
1180,Spotify,"Machine Learning Engineer, Personalization, Mi...",lever,old,2,74,Good Match,False
1179,Spotify,"Machine Learning Engineer I, Personalization ,...",lever,old,2,74,Good Match,False
1231,Spotify,"Senior Machine Learning Engineer, Zeitgeist, P...",lever,old,3,70,Good Match,False
1176,Spotify,Machine Learning Engineer,lever,old,3,70,Good Match,False
1232,Spotify,"Senior Machine Learning Engineer, Zeitgeist, P...",lever,old,3,70,Good Match,False
1079,Workday,Senior Software Engineer (Gen AI),workday,fresh,2,70,Good Match,True
310,OpenAI,Research Engineer / Machine Learning Engineer ...,ashby,old,2,67,Good Match,False
1255,Spotify,Staff Machine Learning Engineer,lever,old,3,66,Good Match,False
1254,Spotify,Staff Machine Learning Engineer,lever,old,3,66,Good Match,False
901,Anthropic,"ML Infrastructure Engineer, Safeguards",greenhouse,old,1,63,Maybe,False
